# Analisi Computazionale Avanzata delle Morfologie delle Tracce nei Rivelatori SSNTD: Ricostruzione del Paradigma autoTRAK e Implementazione di Machine Learning Non Supervisionato per la Spettrometria di Energia su CR-39


In [1]:
# ==============================================================================
# Cell 1: Essential Library Imports and Environment Setup
# ==============================================================================
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass
import warnings

# Suppressing minor scikit-learn algorithmic warnings 
warnings.filterwarnings("ignore")

print("Cell 1: Computational modules loaded. Ready for morphological ML and energy spectrometry.")


Cell 1: Computational modules loaded. Ready for morphological ML and energy spectrometry.


In [ ]:
# ==============================================================================
# Cell 2: System Configuration and Physical Hyperparameters
# ==============================================================================
@dataclass
class Config:
    """
    Manages all critical paths, Region of Interest (ROI) slicing coordinates, 
    morphological kernel sizes, and physics-based energy conversion factors.
    """
    data_dir: Path = Path("../data_test")
    output_dir: Path = Path("./outputs")
    # Broadened glob to capture any jpg if standard naming varies
    image_glob: str = "*.jpg" 
    
    # ROI configuration
    roi_x: int = 2188
    roi_y: int = 2244
    roi_w: int = 5072
    roi_h: int = 4960
    
    # Morphological Welding & Noise Rejection Parameters
    intensity_threshold: int = 0
    closing_kernel_size: int = 5
    min_area: float = 20.0
    max_area: float = 10000.0
    
    # GMM Settings
    n_clusters: int = 4
    random_state: int = 42
    
    # Watershed Topographical Splitter Settings
    ws_enabled: bool = True
    ws_min_area: float = 160.0
    ws_min_fragment_area: float = 20.0 
    ws_peak_min_distance: int = 3
    
    # Energy Calibration Parameters
    max_theoretical_depth_um: float = 15.0  
    bg_mode_intensity: float = 10.0         
    energy_conversion_factor: float = 0.08  

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
print(f"Cell 2: Hyperparameters loaded. Target data directory: {cfg.data_dir.absolute()}")


Cell 2: Hyperparameters loaded. Target data directory: /Users/fc/Library/CloudStorage/OneDrive-MuseoStoricodellaFisicaeCentroStudieRicercheEnricoFermi/acme/CREF/sourceforge/github/radon/algorithms/../data_test


In [3]:
# ==============================================================================
# Cell 3: Morphological Closing and Feature/Energy Extraction
# ==============================================================================
def apply_morphological_welding(binary_image, kernel_size):
    """
    Executes a morphological closing (Dilation -> Erosion) operation.
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    welded_image = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel)
    return welded_image

def calculate_energy_from_greyscale(max_intensity, cfg):
    """
    Mathematically implements the autoTRAK greyscale-depth methodology.
    """
    delta_i = abs(max_intensity - cfg.bg_mode_intensity)
    if delta_i < 1:
        return 0.0
    
    estimated_depth_um = (cfg.max_theoretical_depth_um / 255.0) * delta_i
    estimated_energy_mev = estimated_depth_um * cfg.energy_conversion_factor
    return estimated_energy_mev

def extract_features_from_roi(image_path, cfg):
    """
    Full extraction pipeline: reads image, crops to ROI safely, creates binary mask, 
    welds fragments, extracts shapes, and logs multidimensional features.
    """
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None, None, None
        
    # Safe crop checking image bounds to prevent silent empty arrays
    h, w = img.shape
    roi_y_end = min(cfg.roi_y + cfg.roi_h, h)
    roi_x_end = min(cfg.roi_x + cfg.roi_w, w)
    roi = img[cfg.roi_y:roi_y_end, cfg.roi_x:roi_x_end]
    
    _, binary = cv2.threshold(roi, cfg.intensity_threshold, 255, cv2.THRESH_BINARY)
    welded_binary = apply_morphological_welding(binary, cfg.closing_kernel_size)
    contours, _ = cv2.findContours(welded_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    features = []
    valid_contours = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < cfg.min_area or area > cfg.max_area:
            continue
            
        perimeter = cv2.arcLength(cnt, True)
        circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        x, y, w_b, h_b = cv2.boundingRect(cnt)
        aspect_ratio = float(w_b) / h_b if h_b > 0 else 0
        
        mask = np.zeros_like(roi)
        cv2.drawContours(mask, [cnt], -1, 255, -1)
        contour_pixels = roi[mask == 255]
        max_intensity = np.max(contour_pixels) if len(contour_pixels) > 0 else 0
        
        energy_mev = calculate_energy_from_greyscale(max_intensity, cfg)
        
        features.append([area, perimeter, circularity, aspect_ratio, max_intensity, energy_mev])
        valid_contours.append(cnt)
        
    return np.array(features) if features else None, valid_contours, roi, welded_binary

print("Cell 3: Morphological and Greyscale energy extraction active.")


Cell 3: Morphological and Greyscale energy extraction active.


In [4]:
# ==============================================================================
# Cell 4: Watershed Segmentation for Touching/Merged Tracks
# ==============================================================================
def split_contour_watershed(roi, cnt, cfg):
    """
    Applies distance-transform based watershed segmentation to slice apart 
    overlapping physical tracks that were merged into a single digital contour.
    """
    x, y, w_b, h_b = cv2.boundingRect(cnt)
    local_roi = roi[y:y+h_b, x:x+w_b]
    
    local_mask = np.zeros((h_b, w_b), dtype=np.uint8)
    shifted_cnt = cnt - [x, y]
    cv2.drawContours(local_mask, [shifted_cnt], -1, 255, -1)
    
    distance = ndimage.distance_transform_edt(local_mask)
    coords = peak_local_max(distance, min_distance=cfg.ws_peak_min_distance, labels=local_mask)
    
    if len(coords) <= 1:
        return [cnt]
        
    mask_bool = local_mask.astype(bool)
    markers = np.zeros_like(local_mask, dtype=np.int32)
    for i, (r, c) in enumerate(coords):
        markers[r, c] = i + 1
        
    labels = watershed(-distance, markers, mask=mask_bool)
    split_contours = []
    
    for label_idx in range(1, len(coords) + 1):
        fragment_mask = np.uint8(labels == label_idx) * 255
        frags, _ = cv2.findContours(fragment_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for frag in frags:
            global_frag = frag + [x, y]
            if cv2.contourArea(global_frag) >= cfg.ws_min_fragment_area:
                split_contours.append(global_frag)
                
    return split_contours

print("Cell 4: Watershed topology engine initialized.")

Cell 4: Watershed topology engine initialized.


In [5]:
# ==============================================================================
# Cell 5: Unsupervised Gaussian Mixture Model (GMM) Classification
# ==============================================================================
def run_gmm_pipeline(features_matrix, cfg):
    """
    Utilizes Gaussian Mixture Models with full covariance matrices to cluster 
    events stochastically.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_matrix[:, :4])
    
    gmm = GaussianMixture(
        n_components=cfg.n_clusters, 
        covariance_type='full', 
        random_state=cfg.random_state
    )
    labels = gmm.fit_predict(X_scaled)
    
    cluster_stats = []
    for k in range(cfg.n_clusters):
        cluster_mask = (labels == k)
        if not np.any(cluster_mask):
            continue
            
        mean_area = np.mean(features_matrix[cluster_mask, 0])
        mean_circ = np.mean(features_matrix[cluster_mask, 2])
        mean_intensity = np.mean(features_matrix[cluster_mask, 4])
        
        saliency = mean_area * mean_intensity
        cluster_stats.append({
            'Cluster_ID': k,
            'Count': np.sum(cluster_mask),
            'Mean_Area': mean_area,
            'Mean_Circularity': mean_circ,
            'Saliency': saliency
        })
        
    stats_df = pd.DataFrame(cluster_stats)
    return labels, gmm, scaler, stats_df

print("Cell 5: GMM Unsupervised Clustering pipeline compiled.")


Cell 5: GMM Unsupervised Clustering pipeline compiled.


In [6]:
# ==============================================================================
# Cell 6: Execution Pipeline and CSV Database Export
# ==============================================================================
image_paths = list(cfg.data_dir.glob(cfg.image_glob))

print("Initiating exhaustive morpho-optical scanning protocol...")
print(f"Working Directory: {cfg.data_dir.absolute()}")
print(f"Images detected: {len(image_paths)}")

all_features = []
all_contours = []

for path in image_paths:
    if not path.is_file():
        continue
        
    print(f"  Scanning target: {path.name}...")
    feats, cnts, roi, binary = extract_features_from_roi(path, cfg)
    
    if feats is not None and len(feats) > 0:
        print(f"    -> Phase 1 Extraction: {len(feats)} raw events localized.")
        
        if cfg.ws_enabled:
            refined_feats = []
            refined_cnts = []
            for i, c in enumerate(cnts):
                if feats[i, 0] > cfg.ws_min_area:
                    split_cnts = split_contour_watershed(roi, c, cfg)
                    for sc in split_cnts:
                        area = cv2.contourArea(sc)
                        perimeter = cv2.arcLength(sc, True)
                        circ = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
                        x, y, w_b, h_b = cv2.boundingRect(sc)
                        asp = float(w_b) / h_b if h_b > 0 else 0
                        
                        mask = np.zeros_like(roi)
                        cv2.drawContours(mask, [sc], -1, 255, -1)
                        c_pix = roi[mask == 255]
                        max_int = np.max(c_pix) if len(c_pix) > 0 else 0
                        energy = calculate_energy_from_greyscale(max_int, cfg)
                        
                        refined_feats.append([area, perimeter, circ, asp, max_int, energy])
                        refined_cnts.append(sc)
                else:
                    refined_feats.append(feats[i])
                    refined_cnts.append(c)
                    
            feats = np.array(refined_feats)
            cnts = refined_cnts
            print(f"    -> Phase 2 Watershed Refinement: {len(feats)} total distinct events.")
        
        all_features.append(feats)
        all_contours.extend(cnts)
    else:
        print("    -> No events localized. Image might be blank or ROI bounds are invalid.")

if all_features:
    X_matrix = np.vstack(all_features)
    print("\nExecuting GMM Clustering mapping in multidimensional space...")
    
    labels, gmm_model, scaler, cluster_summary = run_gmm_pipeline(X_matrix, cfg)
    
    print("\n=== Morphological Cluster Statistics ===")
    print(cluster_summary.to_string(index=False))
    
    final_db = pd.DataFrame(X_matrix, columns=['Area', 'Perimeter', 'Circularity', 'Aspect_Ratio', 'Max_Intensity', 'Energy_MeV'])
    final_db['Cluster_ID'] = labels
    
    output_csv_path = cfg.output_dir / "gmm_blobs.csv"
    final_db.to_csv(output_csv_path, index=False)
    
    signal_mask = final_db['Circularity'] > 0.8
    if signal_mask.any():
        mean_energy = final_db[signal_mask]['Energy_MeV'].mean()
        print(f"\nEstimated Mean Kinetic Energy of Valid Signal Tracks: {mean_energy:.3f} MeV")
    else:
        print("\nNo signal tracks with circularity > 0.8 found.")
        
    print(f"Pipeline Finalized. Event database exported to: {output_csv_path.absolute()}")
else:
    print("\nPipeline Finalized. No valid features extracted. Verify 'data_dir' path and image availability.")


Initiating exhaustive morpho-optical scanning protocol...
Working Directory: /Users/fc/Library/CloudStorage/OneDrive-MuseoStoricodellaFisicaeCentroStudieRicercheEnricoFermi/acme/CREF/sourceforge/github/radon/algorithms/../data_test
Images detected: 15
  Scanning target: LBS255624.jpg...
    -> Phase 1 Extraction: 959 raw events localized.
    -> Phase 2 Watershed Refinement: 1031 total distinct events.
  Scanning target: LBS255618.jpg...
    -> Phase 1 Extraction: 767 raw events localized.
    -> Phase 2 Watershed Refinement: 787 total distinct events.
  Scanning target: LBS255619.jpg...
    -> Phase 1 Extraction: 863 raw events localized.
    -> Phase 2 Watershed Refinement: 905 total distinct events.
  Scanning target: LBS255625.jpg...
    -> Phase 1 Extraction: 905 raw events localized.
    -> Phase 2 Watershed Refinement: 932 total distinct events.
  Scanning target: LBS255622.jpg...
    -> Phase 1 Extraction: 978 raw events localized.
    -> Phase 2 Watershed Refinement: 1013 tota